In [1]:
import pandas as pd, json, hashlib

decisions = pd.read_csv("../data/processed/decisions.csv")

log = []
prev_hash = "0" * 64   # pierwszy wpis nie ma poprzednika, więc same zera

for _, row in decisions.iterrows():
    entry = {
        "customer_id": row["customer_id"],
        "decision": row["decision"],
        "rule": row["rule"],
        "confidence": float(row["confidence"]),
        "rationale": row["rationale"],
        "model": "claude-haiku-5-5",
        "prev_hash": prev_hash,
    }
    text = json.dumps(entry, sort_keys=True, ensure_ascii=False)
    entry["hash"] = hashlib.sha256(text.encode()).hexdigest()
    log.append(entry)
    prev_hash = entry["hash"]

with open("../data/processed/audit_log.jsonl", "w") as f:
    for e in log:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")

print(len(log), "wpisów; ostatni hash:", prev_hash[:16], "...")

21 wpisów; ostatni hash: 3d272c79818beefd ...


In [2]:
def verify(log):
    prev = "0" * 64
    for i, e in enumerate(log):
        body = {k: v for k, v in e.items() if k != "hash"}
        if body["prev_hash"] != prev:
            return f"❌ Łańcuch przerwany przy wpisie {i}"
        text = json.dumps(body, sort_keys=True, ensure_ascii=False)
        if hashlib.sha256(text.encode()).hexdigest() != e["hash"]:
            return f"❌ Wpis {i} został zmieniony"
        prev = e["hash"]
    return "✅ Dziennik nienaruszony"

print(verify(log))

✅ Dziennik nienaruszony


In [3]:
import copy
fake = copy.deepcopy(log)
fake[3]["decision"] = "FALSE_POSITIVE"   # ktoś „zamiata” prawdziwe trafienie
print(verify(fake))

❌ Wpis 3 został zmieniony
